### RAG Application Using Type Sense

In [1]:
import typesense

In [2]:
client = typesense.Client({
    'nodes': [{
        'host': 'pyco2al6701wzxgjp-1.a2.typesense.net',
        'port': 443,
        'protocol': 'https'
    }],
    'api_key': 'VEXe1Nf5S2cyfmJFFpOQU5e6MRUlakpS',
    'connection_timeout_seconds': 2
})

books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'authors', 'type': 'string[]', 'facet': True},
        {'name': 'publication_year', 'type': 'int32'},
        {'name': 'ratings_count', 'type': 'int32'},
        {'name': 'average_rating', 'type': 'float'},
    ],
    'default_sorting_field': 'ratings_count'
}

try:
    print(client.collections.create(books_schema))
except typesense.exceptions.ObjectAlreadyExists:
    print("Collection 'books' already exists. Deleting and recreating with updated schema...")
    client.collections['books'].delete()
    print(client.collections.create(books_schema))

Collection 'books' already exists. Deleting and recreating with updated schema...
{'created_at': 1775979011, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': ''

In [3]:
client

In [4]:
with open('book.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [5]:
search_parameters = {
    'q': 'harry potter',
    'query_by': 'title,authors',
    'sort_by': 'ratings_count:desc'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [6]:
search_parameters = {
    'q': 'harry potter',
    'query_by': 'title,authors',
    'filter_by': 'publication_year:>2000',
    'sort_by': 'ratings_count:desc'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 11,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.61,
    'id': '25',
    'image_url': 'https://images.gr-assets.com/books/1474171184m/136251.jpg',
    'publication_year': 2007,
    'ratings_count': 1746574,
    'title': 'Harry Potter and the Deathly Hallows'},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Deathly Hallows'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Deathly Hallows'}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'ave

In [7]:
search_parameters = {
    'q': 'experyment',
    'query_by': 'title',
    'facet_by': 'authors',
    'sort_by': 'ratings_count:desc'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': ' Käthe Mazur',
     'value': ' Käthe Mazur'},
    {'count': 1, 'highlighted': 'Mahatma Gandhi', 'value': 'Mahatma Gandhi'},
    {'count': 1, 'highlighted': 'Gretchen Rubin', 'value': 'Gretchen Rubin'},
    {'count': 1,
     'highlighted': 'James Patterson',
     'value': 'James Patterson'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 4}}],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152.jpg',
    'publication_year': 2005,
    'ratings_count': 172302,
    'title': 'The Angel Experiment'},
   'highlight': {'title': {'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}],
   'text_match': 5787300

In [8]:
### Langchain + Groq LLM + Typesense + RAG Application
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

e:\Agentic AI\RAGS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [14]:
loader = TextLoader('Test.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

Created a chunk of size 646, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 836, which is longer than the specified 500
Created a chunk of size 837, which is longer than the specified 500
Created a chunk of size 837, which is longer than the specified 500
Created a chunk of size 837, which is longer than the specified 500
Created a chunk of size 837, which is longer than the specified 500
Created a chunk of size 837, which is longer than the specified 500
Created a chunk of size 837, which is longer tha

In [15]:
docsearch = Typesense.from_documents(
    docs, 
    embeddings, 
    typesense_client_params={
        "host": "pyco2al6701wzxgjp-1.a2.typesense.net",
        "port": 443,
        "protocol": "https",
        "typesense_api_key": "VEXe1Nf5S2cyfmJFFpOQU5e6MRUlakpS",
        "typesense_collection_name": "lang-chain",
        "connection_timeout_seconds": 30
    },
    
)

In [16]:
query = "What is artificial intelligence?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

In conclusion, Artificial Intelligence is a powerful technology that is
shaping the future of humanity. While it offers numerous advantages, it
also requires responsible implementation to ensure it benefits society
as a whole. 305. Artificial Intelligence (AI) is transforming the modern
world by reshaping industries, enhancing human capabilities, and
redefining how we interact with technology. From healthcare to finance,
education to transportation, AI systems are increasingly integrated into
everyday life. AI refers to machines or software that can perform tasks
that typically require human intelligence, such as learning, reasoning,
problem-solving, and decision-making. Over the years, AI has evolved
from simple rule-based systems to advanced machine learning and deep
learning models capable of analyzing vast amounts of data.


In [17]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x0000029522A78B90>, search_kwargs={})

In [18]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'Test.txt'}, page_content='In conclusion, Artificial Intelligence is a powerful technology that is\nshaping the future of humanity. While it offers numerous advantages, it\nalso requires responsible implementation to ensure it benefits society\nas a whole. 630. Artificial Intelligence (AI) is transforming the modern\nworld by reshaping industries, enhancing human capabilities, and\nredefining how we interact with technology. From healthcare to finance,\neducation to transportation, AI systems are increasingly integrated into\neveryday life. AI refers to machines or software that can perform tasks\nthat typically require human intelligence, such as learning, reasoning,\nproblem-solving, and decision-making. Over the years, AI has evolved\nfrom simple rule-based systems to advanced machine learning and deep\nlearning models capable of analyzing vast amounts of data.')